# 02 — Tier 2 Live Pipeline Latency Benchmark
Executes the live pipeline benchmark measuring physical Onset and Offset/Hangover Latency anchored to calibration marker T0.

In [1]:
# Cell 1: Environment Setup & Directory Anchor
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root anchored at: {Path.cwd()}')

Project root anchored at: /content/voice-bot-vad-harness


In [2]:
# Cell 2: Execute Tier 2 Benchmark
import json
from orchestrator import run_benchmark

config_path = Path('config/run_config.yaml')
print(f'Executing Tier 2 Benchmark using config: {config_path.resolve()}')
run_summary = run_benchmark(config_path=config_path)

print('\n--- Tier 2 Execution Summary ---')
print(json.dumps(run_summary, indent=2))

Executing Tier 2 Benchmark using config: /content/voice-bot-vad-harness/config/run_config.yaml
2026-09-20 06:43:39,550 [INFO] [orchestrator] Loaded run configuration for run_id: 'vad_eval_run_20260913_001'


INFO:orchestrator:Loaded run configuration for run_id: 'vad_eval_run_20260913_001'


2026-09-20 06:43:39,557 [INFO] [orchestrator] Loaded manifest 'corpus/manifest.json' (229 clips, hash=8fa135caac7e0511)


INFO:orchestrator:Loaded manifest 'corpus/manifest.json' (229 clips, hash=8fa135caac7e0511)


2026-09-20 06:43:39,560 [INFO] [orchestrator] === Starting Tier 2 Live Pipeline Evaluation [Run ID: vad_eval_run_20260913_001] ===


...

2026-09-20 06:43:41,281 [ERROR] [orchestrator] [Run vad_eval_run_20260913_001 | Tier 2] Fatal tier failure: Failed to join Jitsi room 'vad-benchmark-eval-room-01': Failed to join Jitsi room 'None': Failed to connect signaling session: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1032)
Traceback (most recent call last):
  File "/content/voice-bot-vad-harness/webrtc_bridge/jitsi_signaling.py", line 356, in connect
    self._ws = await websockets.connect(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/websockets/asyncio/client.py", line 541, in __await_impl__
    self.connection = await self.create_connection()
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/websockets/asyncio/client.py", line 467, in create_connection
    _, connection = await loop.create_connection(factory, **kwargs)
                    ^^^^^^^^^^^^

/content/voice-bot-vad-harness/webrtc_bridge/media_session.py:446: RuntimeWarning: coroutine 'BridgeMediaSession._async_close' was never awaited
  logger.debug("Notice closing peer connection: %s", exc)
ERROR:orchestrator:[Run vad_eval_run_20260913_001 | Tier 2] Fatal tier failure: Failed to join Jitsi room 'vad-benchmark-eval-room-01': Failed to join Jitsi room 'None': Failed to connect signaling session: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1032)
Traceback (most recent call last):
  File "/content/voice-bot-vad-harness/webrtc_bridge/jitsi_signaling.py", line 356, in connect
    self._ws = await websockets.connect(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/websockets/asyncio/client.py", line 541, in __await_impl__
    self.connection = await self.create_connection()
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/d

2026-09-20 06:43:41,288 [INFO] [orchestrator] === Benchmark Run 'vad_eval_run_20260913_001' Completed Successfully ===


INFO:orchestrator:=== Benchmark Run 'vad_eval_run_20260913_001' Completed Successfully ===



--- Tier 2 Execution Summary ---
{
  "run_id": "vad_eval_run_20260913_001",
  "manifest_version": "8fa135caac7e0511",
  "tiers_executed": {
    "tier2": {
      "status": "failed",
      "error": "Failed to join Jitsi room 'vad-benchmark-eval-room-01': Failed to join Jitsi room 'None': Failed to connect signaling session: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1032)"
    }
  }
}


In [3]:
# Cell 3: Display Latency & Evaluation Records
from pathlib import Path
from storage.results_store import ResultsStore

results_dir = Path('results/run_output')
store = ResultsStore(results_dir)

run_id = run_summary.get('run_id', 'vad_eval_run_20260913_001')
records = list(store.read_raw_records(run_id=run_id))

print(f'Total durable evaluation records persisted: {len(records)}\n')
for r in records[:10]:
    print(f"[Adapter: {r['adapter_id']:<18}] [Clip: {r['clip_id']:<25}] [Type: {r['record_type']:<22}]")
    if 'latency' in r['record_type']:
        print(f"   --> Payload: {r['payload']}")

store.close()

Total durable evaluation records persisted: 0

